# Paper 4 – Combine the Nepal Historical Weather Dataset

This notebook downloads all 8 original Parquet shards from `Prazzwal07/local_weather_data` and combines them into one file: `Nepal_Weather_Combined.parquet`.

**Source size:** 21,506,175 rows, approximately 1.47 GB. Keep the result as Parquet rather than CSV because it is a large dataset.

Run the next cell. You need a stable internet connection and enough free disk space (allow several GB for the downloaded shards and output).

In [ ]:
# Combine the 8 original Parquet shards of Paper 4's local_weather_data
# Source: https://huggingface.co/datasets/Prazzwal07/local_weather_data
#
# Output: Nepal_Weather_Combined.parquet
# The source dataset contains 21,506,175 rows and is about 1.47 GB.
# This script keeps the data in Parquet format (recommended for this size).

import sys
import subprocess
import importlib.util
from pathlib import Path

def ensure_package(import_name, package_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package_name or import_name
        ])

ensure_package("huggingface_hub")
ensure_package("pyarrow")

from huggingface_hub import snapshot_download
import pyarrow as pa
import pyarrow.parquet as pq

REPO_ID = "Prazzwal07/local_weather_data"
OUTPUT_FILE = Path("Nepal_Weather_Combined.parquet")
BATCH_SIZE = 100_000

print("Step 1/3: Downloading the original Parquet shards from Hugging Face...")
repo_dir = Path(snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=["data/*.parquet"],
))
data_dir = repo_dir / "data"
shard_files = sorted(data_dir.glob("train-*.parquet"))

if not shard_files:
    raise FileNotFoundError(
        f"No train-*.parquet files found in {data_dir}. "
        "Open the dataset page and check its file structure."
    )

print(f"Found {len(shard_files)} original shard files:")
for f in shard_files:
    print(f"  - {f.name} ({f.stat().st_size / (1024**2):.1f} MB)")

print("\nStep 2/3: Combining shards into one Parquet file (streaming batches)...")
writer = None
total_rows = 0

try:
    for shard_index, shard_path in enumerate(shard_files, start=1):
        parquet_file = pq.ParquetFile(shard_path)
        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_FILE,
                parquet_file.schema_arrow,
                compression="zstd",
                use_dictionary=True,
            )
        shard_rows = 0
        for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
            writer.write_batch(batch)
            rows = batch.num_rows
            shard_rows += rows
            total_rows += rows
        print(f"  {shard_index}/{len(shard_files)}: {shard_path.name} -> {shard_rows:,} rows")
finally:
    if writer is not None:
        writer.close()

if writer is None:
    raise RuntimeError("No Parquet data was written.")

print("\nStep 3/3: Verifying the combined file...")
combined = pq.ParquetFile(OUTPUT_FILE)
print(f"Output file : {OUTPUT_FILE.resolve()}")
print(f"Rows        : {combined.metadata.num_rows:,}")
print(f"Columns     : {len(combined.schema_arrow.names)}")
print(f"Column names: {combined.schema_arrow.names}")
print(f"File size   : {OUTPUT_FILE.stat().st_size / (1024**3):.2f} GB")
print("\nPreview (first 5 rows):")
print(combined.read_row_group(0).slice(0, 5).to_pandas().to_string(index=False))

assert combined.metadata.num_rows == total_rows, "Row-count verification failed!"
print("\nSUCCESS: All original shards are now combined into one Parquet file.")
